In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"

# Analysis

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

grangercausalitytests(df_part3[['sentiment', 'vix_ln_return']], maxlag=2)#, verbose=False)

In [ ]:
import matplotlib.pyplot as plt
import causalimpact
from statsmodels.tsa.arima_process import ArmaProcess
from causalimpact import CausalImpact

import datetime
# df["sentiment"] =df["sentiment"].fillna(0)
# df["vix"] =df["vix"].fillna(0)
# df_part3["sp500"] =df["sp500"].fillna(0)
df_part3.fillna(0, inplace=True)
data = df_part3[["sp500","sentiment"]].values.tolist()

pre_period = [1,200]
post_period = [201,255]
ci = CausalImpact(data, pre_period, post_period)
print(ci.summary())
print(ci.summary(output='report'))
ci.plot()

# Causal Impact

In [ ]:
tsdata_to_autocov(df["sentiment"], df["vix"])

10. How to test for stationarity?
The stationarity of a series can be established by looking at the plot of the series like we did earlier.

Another method is to split the series into 2 or more contiguous parts and computing the summary statistics like the mean, variance and the autocorrelation. If the stats are quite different, then the series is not likely to be stationary.

Nevertheless, you need a method to quantitatively determine if a given series is stationary or not. This can be done using statistical tests called ‘Unit Root Tests’. There are multiple variations of this, where the tests check if a time series is non-stationary and possess a unit root.

There are multiple implementations of Unit Root tests like:

1. Augmented Dickey Fuller test (ADH Test)
2. Kwiatkowski-Phillips-Schmidt-Shin – KPSS test (trend stationary)
3. Philips Perron test (PP Test)
The most commonly used is the ADF test, where the null hypothesis is the time series possesses a unit root and is non-stationary. So, id the P-Value in ADH test is less than the significance level (0.05), you reject the null hypothesis.

The KPSS test, on the other hand, is used to test for trend stationarity. The null hypothesis and the P-Value interpretation is just the opposite of ADH test. The below code implements these two tests using statsmodels package in python.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
df_vix_ln_return = pd.DataFrame()
df_vix_ln_return = df_part3[["vix_ln_return"]]
print(df_vix_ln_return)
# ADF Test
result = adfuller(df_vix_ln_return.vix_ln_return.values, autolag='AIC')
print(f'ADF Statistic: {result[0]}')
print(f'p-value: {result[1]}')
for key, vix_ln_return in result[4].items():
    print('Critial Values:')
    print(f'   {key}, {vix_ln_return}')

# KPSS Test
result = kpss(df_vix_ln_return.vix_ln_return.values, regression='c')
print('\nKPSS Statistic: %f' % result[0])
print('p-value: %f' % result[1])
for key, vix_ln_return in result[3].items():
    print('Critial Values:')
    print(f'   {key}, {vix_ln_return}')


In [ ]:
grangercausalitytests(df[['sentiment', 'month']], maxlag=2)

16. What is autocorrelation and partial autocorrelation functions?
Autocorrelation is simply the correlation of a series with its own lags. If a series is significantly autocorrelated, that means, the previous values of the series (lags) may be helpful in predicting the current value.
Partial Autocorrelation also conveys similar information but it conveys the pure correlation of a series and its lag, excluding the correlation contributions from the intermediate lags.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/a10.csv')

# Calculate ACF and PACF upto 50 lags
# acf_50 = acf(df.value, nlags=50)
# pacf_50 = pacf(df.value, nlags=50)

# Draw Plot
fig, axes = plt.subplots(1,2,figsize=(16,3), dpi= 100)
plot_acf(df.value.tolist(), lags=50, ax=axes[0])
plot_pacf(df.value.tolist(), lags=50, ax=axes[1])

17. How to compute partial autocorrelation function?
So how to compute partial autocorrelation?

The partial autocorrelation of lag (k) of a series is the coefficient of that lag in the autoregression equation of Y. The autoregressive equation of Y is nothing but the linear regression of Y with its own lags as predictors.

For Example, if Y_t is the current series and Y_t-1 is the lag 1 of Y, then the partial autocorrelation of lag 3 (Y_t-3) is the coefficient $\alpha_3$ of Y_t-3 in the following equation: